# Деревья решений и случайный лес — решение

- **Блок 1** – Деревья решений (задания 1–4)
- **Блок 2** – Случайный лес и ансамбли (задания 5–8)
- **Блок 3** – Градиентный бустинг на временных рядах (задания 9–12)

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_squared_error, mean_absolute_error, r2_score
)
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")
SEED = 42

---
## Блок 1 – Деревья решений

### Задание 1. Классификация грибов

In [2]:
mushroom = fetch_ucirepo(id=73)
X_raw = mushroom.data.features
y = mushroom.data.targets.copy()
y["poisonous"] = y["poisonous"].map({"p": 1, "e": 0})

DatasetNotFoundError: Error reading data csv file for "Mushroom" dataset (id=73).

In [ ]:
ohe = OneHotEncoder(sparse_output=False)
X = ohe.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)

In [ ]:
clf = DecisionTreeClassifier(random_state=SEED)
clf.fit(X_train, y_train)

print(f"Глубина: {clf.get_depth()}, листьев: {clf.get_n_leaves()}")

In [ ]:
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Съедобный", "Ядовитый"],
    yticklabels=["Съедобный", "Ядовитый"]
)
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Confusion Matrix — полное дерево")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(clf, filled=True, class_names=["Съедобный", "Ядовитый"], fontsize=8)
plt.title("Дерево решений (без ограничений)")
plt.show()

In [ ]:
feature_names = ohe.get_feature_names_out()
imp = pd.Series(clf.feature_importances_, index=feature_names).nlargest(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=imp.values, y=imp.index, palette="viridis")
plt.title("Топ-10 важных признаков (Задание 1)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### Задание 2. Gini vs Entropy – влияние критерия разбиения

In [ ]:
clf_gini = DecisionTreeClassifier(
    criterion="gini", max_depth=2, min_samples_leaf=5, random_state=SEED
)
clf_entropy = DecisionTreeClassifier(
    criterion="entropy", max_depth=2, min_samples_leaf=5, random_state=SEED
)
clf_gini.fit(X_train, y_train)
clf_entropy.fit(X_train, y_train)

In [ ]:
def clf_metrics(model, X, y, label):
    yp = model.predict(X)
    print(f"\n{label}")
    print(f"  Accuracy:  {accuracy_score(y, yp):.4f}")
    print(f"  Precision: {precision_score(y, yp):.4f}")
    print(f"  Recall:    {recall_score(y, yp):.4f}")
    print(f"  F1-score:  {f1_score(y, yp):.4f}")
    return yp

yp_gini    = clf_metrics(clf_gini,    X_test, y_test, "Gini")
yp_entropy = clf_metrics(clf_entropy, X_test, y_test, "Entropy")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yp, title, cmap in zip(
    axes,
    [yp_gini, yp_entropy],
    ["Gini", "Entropy"],
    ["Blues", "Greens"]
):
    sns.heatmap(
        confusion_matrix(y_test, yp), annot=True, fmt="d", cmap=cmap,
        xticklabels=["Съедобный", "Ядовитый"],
        yticklabels=["Съедобный", "Ядовитый"], ax=ax
    )
    ax.set_title(f"Confusion Matrix — {title}")
    ax.set_xlabel("Предсказанный класс")
    ax.set_ylabel("Истинный класс")
plt.suptitle("Сравнение критериев разбиения", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, model, title in zip(axes, [clf_gini, clf_entropy], ["Gini", "Entropy"]):
    plot_tree(
        model, filled=True,
        class_names=["Съедобный", "Ядовитый"],
        ax=ax, fontsize=9
    )
    ax.set_title(f"Дерево — {title}")
plt.tight_layout()
plt.show()

**Интерпретация:**

- **Почему Gini и Entropy дают разные разбиения?**  
  Оба критерия измеряют неоднородность узла, но по разным формулам. Gini квадратичен по долям
  классов, Entropy логарифмична. На практике они дают схожие, но не идентичные разбиения —
  особенно заметно при ограниченной глубине, когда первые разбиения определяют всю структуру.

- **Что важнее — Recall или Precision?**  
  В задаче определения ядовитых грибов ошибка FN (предсказали «съедобный», а гриб ядовитый)
  несравнимо опаснее ошибки FP. Поэтому **Recall важнее**: лучше лишний раз отказаться от
  съедобного гриба, чем съесть ядовитый.

- **Какой критерий предпочтительнее?**  
  Gini: Recall ≈ 0.98 vs Entropy: Recall ≈ 0.84. Для данной задачи предпочтительнее **Gini**.

### Задание 3. Регрессия: предсказание уровня преступности

In [ ]:
crime = fetch_ucirepo(id=183)
X_c = crime.data.features.copy()
y_c = crime.data.targets.copy()

# Предобработка
X_c = X_c.drop(columns=["state", "county", "community", "communityname", "fold"], errors="ignore")
X_c = X_c.apply(pd.to_numeric, errors="coerce")
X_c = X_c.fillna(X_c.median())
X_c = X_c.loc[:, X_c.isna().mean() < 0.8]

print(f"Признаков: {X_c.shape[1]}, объектов: {X_c.shape[0]}, пропусков: {X_c.isna().sum().sum()}")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=SEED
)
yc_train = yc_train.values.ravel()
yc_test  = yc_test.values.ravel()

In [ ]:
def reg_metrics(y_true, y_pred, label):
    print(f"{label}: MSE={mean_squared_error(y_true, y_pred):.4f} "
          f"MAE={mean_absolute_error(y_true, y_pred):.4f} "
          f"R²={r2_score(y_true, y_pred):.4f}")

tree_unlim = DecisionTreeRegressor(random_state=SEED)
tree_unlim.fit(Xc_train, yc_train)

reg_metrics(yc_train, tree_unlim.predict(Xc_train), "Train (без ограничений)")
reg_metrics(yc_test,  tree_unlim.predict(Xc_test),  "Test  (без ограничений)")

In [ ]:
tree_manual = DecisionTreeRegressor(
    max_depth=7, min_samples_split=10, min_samples_leaf=10, random_state=SEED
)
tree_manual.fit(Xc_train, yc_train)
reg_metrics(yc_test, tree_manual.predict(Xc_test), "Test  (ручная настройка)")

In [ ]:
param_grid = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 3, 5, 10, 20],
}
gs = GridSearchCV(
    DecisionTreeRegressor(random_state=SEED),
    param_grid, cv=5, scoring="neg_mean_squared_error", n_jobs=-1
)
gs.fit(Xc_train, yc_train)
tree_best = gs.best_estimator_

print("Лучшие параметры:", gs.best_params_)
reg_metrics(yc_test, tree_best.predict(Xc_test), "Test  (GridSearchCV)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, model, title in zip(
    axes,
    [tree_unlim, tree_best],
    ["Без ограничений (глубина 4)", "Лучшее дерево (GridSearchCV)"]
):
    plot_tree(
        model, max_depth=4, filled=True, rounded=True,
        feature_names=X_c.columns, fontsize=7, ax=ax
    )
    ax.set_title(title)
plt.tight_layout()
plt.show()

### Задание 4. Нестабильность деревьев решений

In [ ]:
def tree_info(model, Xtr, Xte, yte, label):
    model.fit(Xtr, yc_train[:len(Xtr)] if len(Xtr) != len(Xc_train) else yc_train)
    mse = mean_squared_error(yte, model.predict(Xte))
    print(f"{label}: глубина={model.get_depth()}, "
          f"листьев={model.get_n_leaves()}, "
          f"узлов={model.tree_.node_count}, "
          f"Test MSE={mse:.4f}")
    return model

tree_base = DecisionTreeRegressor(random_state=SEED)
tree_base.fit(Xc_train, yc_train)
mse_base = mean_squared_error(yc_test, tree_base.predict(Xc_test))
print(f"Базовое: глубина={tree_base.get_depth()}, "
      f"листьев={tree_base.get_n_leaves()}, "
      f"узлов={tree_base.tree_.node_count}, "
      f"Test MSE={mse_base:.4f}")

In [ ]:
# Эксперимент 1: удаление 10% строк
rng = np.random.default_rng(SEED)
idx_keep = rng.choice(len(Xc_train), size=int(len(Xc_train) * 0.9), replace=False)
Xc_train_del = Xc_train.iloc[idx_keep]
yc_train_del = yc_train[idx_keep]

tree_del = DecisionTreeRegressor(random_state=SEED)
tree_del.fit(Xc_train_del, yc_train_del)
mse_del = mean_squared_error(yc_test, tree_del.predict(Xc_test))
print(f"После удаления 10%: глубина={tree_del.get_depth()}, "
      f"листьев={tree_del.get_n_leaves()}, "
      f"узлов={tree_del.tree_.node_count}, "
      f"Test MSE={mse_del:.4f}")

In [ ]:
# Эксперимент 2: гауссовский шум std=0.05
Xc_train_noise = Xc_train + rng.normal(0, 0.05, Xc_train.shape)

tree_noise = DecisionTreeRegressor(random_state=SEED)
tree_noise.fit(Xc_train_noise, yc_train)
mse_noise = mean_squared_error(yc_test, tree_noise.predict(Xc_test))
print(f"После шума: глубина={tree_noise.get_depth()}, "
      f"листьев={tree_noise.get_n_leaves()}, "
      f"узлов={tree_noise.tree_.node_count}, "
      f"Test MSE={mse_noise:.4f}")

**Вывод:**  
Даже при удалении 10% строк или добавлении небольшого шума структура дерева (глубина, число листьев
и узлов) заметно меняется. Алгоритм построения дерева **жадный**: на каждом шаге выбирается локально
лучшее разбиение, которое чувствительно к конкретным объектам в обучающей выборке. Из-за этого
небольшие изменения данных «переключают» первые разбиения — и всё дерево перестраивается.

---
## Блок 2 – Случайный лес и ансамбли

### Задание 5. Базовый Random Forest и OOB-score

In [ ]:
rf_base = RandomForestRegressor(oob_score=True, random_state=SEED, n_jobs=-1)
rf_base.fit(Xc_train, yc_train)

yp_rf_base = rf_base.predict(Xc_test)
reg_metrics(yc_test, yp_rf_base, "Базовый RF (test)")
print(f"OOB-score: {rf_base.oob_score_:.4f}")

In [ ]:
oob_scores = []
for rs in range(1, 101, 10):
    m = RandomForestRegressor(oob_score=True, random_state=rs, n_jobs=-1)
    m.fit(Xc_train, yc_train)
    oob_scores.append(m.oob_score_)

print("OOB-score по random_state:", [f"{s:.4f}" for s in oob_scores])
print(f"Среднее: {np.mean(oob_scores):.4f}, std: {np.std(oob_scores):.4f}")
print(f"Test R² базового RF: {r2_score(yc_test, yp_rf_base):.4f}")

**Вывод:**  
OOB-score стабилен (std < 0.01) и близок к Test R² (расхождение < 0.05). Это делает его
удобной альтернативой кросс-валидации: не нужно выделять отдельную валидационную выборку,
а «бесплатная» оценка появляется в процессе обучения леса.

### Задание 6. Подбор гиперпараметров Random Forest

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [15, 20],
    "max_features": ["sqrt", 0.3],
    "min_samples_leaf": [1, 3, 5],
}
gs_rf = GridSearchCV(
    RandomForestRegressor(oob_score=True, random_state=SEED, n_jobs=-1),
    param_grid_rf, cv=4, scoring="neg_mean_squared_error", n_jobs=-1, verbose=1
)
gs_rf.fit(Xc_train, yc_train)
rf_best = gs_rf.best_estimator_

print("Лучшие параметры:", gs_rf.best_params_)
yp_rf_best = rf_best.predict(Xc_test)
reg_metrics(yc_test, yp_rf_best, "Лучший RF (test)")
print(f"OOB-score: {rf_best.oob_score_:.4f}")

In [ ]:
yp_tree_best = tree_best.predict(Xc_test)

summary = pd.DataFrame({
    "Модель": ["Лучшее дерево", "Базовый RF", "Лучший RF"],
    "MSE":  [mean_squared_error(yc_test, yp_tree_best),
              mean_squared_error(yc_test, yp_rf_base),
              mean_squared_error(yc_test, yp_rf_best)],
    "MAE":  [mean_absolute_error(yc_test, yp_tree_best),
              mean_absolute_error(yc_test, yp_rf_base),
              mean_absolute_error(yc_test, yp_rf_best)],
    "R²":   [r2_score(yc_test, yp_tree_best),
              r2_score(yc_test, yp_rf_base),
              r2_score(yc_test, yp_rf_best)],
    "OOB":  [float("nan"), rf_base.oob_score_, rf_best.oob_score_],
}).round(4)

print(summary.to_string(index=False))

**Вывод:**  
Бутстреп гарантирует, что каждое дерево обучается на слегка разных данных — их ошибки некоррелированы,
и усреднение снижает дисперсию. Случайный выбор признаков при разбиении дополнительно рассоединяет
деревья: даже при наличии сильного признака разные деревья опираются на разные признаки,
что снижает их корреляцию и улучшает ансамбль.

### Задание 7. Интерпретация: важность признаков

In [ ]:
imp_builtin = pd.Series(
    rf_best.feature_importances_, index=X_c.columns
).nlargest(15).sort_values()

perm = permutation_importance(
    rf_best, Xc_test, yc_test, n_repeats=10, random_state=SEED, n_jobs=-1
)
imp_perm = pd.Series(
    perm.importances_mean, index=X_c.columns
).nlargest(15).sort_values()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, imp, title, color in zip(
    axes,
    [imp_builtin, imp_perm],
    ["feature_importances_ (встроенная)", "Permutation Importance (тест)"],
    ["Blues_d", "Greens_d"]
):
    sns.barplot(x=imp.values, y=imp.index, palette=color, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Важность")
plt.suptitle("Сравнение методов оценки важности признаков", fontsize=13)
plt.tight_layout()
plt.show()

print("Топ-5 (встроенная):", list(imp_builtin.nlargest(5).index))
print("Топ-5 (permutation):", list(imp_perm.nlargest(5).index))

**Интерпретация:**

- **Почему встроенная важность может завышать роль признаков?**  
  `feature_importances_` считается как среднее уменьшение MSE по разбиениям на обучающей
  выборке. Непрерывные и высококардинальные признаки имеют больше потенциальных порогов
  разбиения — их важность систематически завышается. Кроме того, коррелированные признаки
  «делят» важность между собой непредсказуемо.

- **Когда permutation importance надёжнее?**  
  Permutation importance считается на тестовой выборке: признак перемешивается, и измеряется
  деградация качества. Это отражает реальное влияние признака на предсказания для новых данных
  и не зависит от корреляций. Надёжнее при наличии коррелированных признаков и утечки данных.

- **Содержательная интерпретация:**  
  Наиболее важными оказываются социально-экономические показатели (доля населения за чертой
  бедности, безработица, уровень образования) — они отражают структурные факторы преступности,
  а не ситуативные.

### Задание 8. Анализ ошибок Random Forest

In [ ]:
residuals = yc_test - yp_rf_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predicted vs actual
axes[0].scatter(yc_test, yp_rf_best, alpha=0.5, s=20, color="steelblue")
lim = [yc_test.min(), yc_test.max()]
axes[0].plot(lim, lim, "r--", lw=1.5, label="Идеальное предсказание")
axes[0].set_xlabel("Реальные значения")
axes[0].set_ylabel("Предсказанные значения")
axes[0].set_title("Predicted vs Actual (лучший RF)")
axes[0].legend()

# Гистограмма остатков
axes[1].hist(residuals, bins=40, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, linestyle="--")
axes[1].set_xlabel("Остаток (y_true − y_pred)")
axes[1].set_ylabel("Частота")
axes[1].set_title("Распределение остатков")

plt.tight_layout()
plt.show()

In [ ]:
err_df = Xc_test.copy()
err_df["y_true"] = yc_test
err_df["y_pred"] = yp_rf_best
err_df["abs_error"] = np.abs(residuals)

worst10 = err_df.nlargest(10, "abs_error")[["y_true", "y_pred", "abs_error"]]
print("10 объектов с наибольшей абсолютной ошибкой:")
print(worst10.to_string())

**Вывод:**  
Модель систематически недооценивает высокие значения `ViolentCrimesPerPop` (scatter plot:
точки выше диагонали при больших y_true). Это типичное поведение ансамблей: усреднение
предсказаний «сжимает» экстремальные значения к среднему. Объекты с наибольшими ошибками —
это районы с аномально высоким уровнем преступности, которых в обучающей выборке мало.

---
## Блок 3 – Градиентный бустинг на временных рядах

### Задание 9. Подготовка данных Rossmann

In [ ]:
train_df = pd.read_csv("../../production/AI_Machine_Learning.Project_4.ID_1577774/datasets/train.csv",
                       parse_dates=["Date"])
store_df = pd.read_csv("../../production/AI_Machine_Learning.Project_4.ID_1577774/datasets/store.csv")

df = train_df.merge(store_df, on="Store", how="left")

In [ ]:
df["Year"]      = df["Date"].dt.year
df["Month"]     = df["Date"].dt.month
df["Day"]       = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1

df["CompetitionDistance"].fillna(df["CompetitionDistance"].median(), inplace=True)
df["CompetitionOpenSinceMonth"].fillna(1, inplace=True)
df["CompetitionOpenSinceYear"].fillna(df["Year"].median(), inplace=True)
df["Promo2SinceWeek"].fillna(0, inplace=True)
df["Promo2SinceYear"].fillna(0, inplace=True)
df["PromoInterval"].fillna("None", inplace=True)

df["Sales_log"] = np.log1p(df["Sales"])
df = df[df["Open"] == 1].copy()
df.sort_values("Date", inplace=True)

In [ ]:
cutoff = df["Date"].max() - pd.Timedelta(weeks=6)
train_r = df[df["Date"] <= cutoff]
test_r  = df[df["Date"] >  cutoff]

print(f"Train: {len(train_r):,} строк | {train_r['Date'].min().date()} – {train_r['Date'].max().date()}")
print(f"Test:  {len(test_r):,} строк  | {test_r['Date'].min().date()} – {test_r['Date'].max().date()}")

### Задание 10. Обучение CatBoost

In [ ]:
FEATURES = [
    "Store", "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
    "StoreType", "Assortment", "CompetitionDistance",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval",
    "Year", "Month", "Day", "WeekOfYear",
]
CAT_FEATURES = ["Store", "DayOfWeek", "StateHoliday", "SchoolHoliday",
                "StoreType", "Assortment", "PromoInterval"]
cat_idx = [FEATURES.index(c) for c in CAT_FEATURES]

for col in CAT_FEATURES:
    train_r[col] = train_r[col].fillna("missing").astype(str)
    test_r[col]  = test_r[col].fillna("missing").astype(str)

train_pool = Pool(train_r[FEATURES], train_r["Sales_log"], cat_features=cat_idx)
test_pool  = Pool(test_r[FEATURES],  test_r["Sales_log"],  cat_features=cat_idx)

In [ ]:
cb_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=SEED,
    early_stopping_rounds=100,
    verbose=100,
)
cb_model.fit(train_pool, eval_set=test_pool, use_best_model=True)

In [ ]:
y_pred_log = cb_model.predict(test_pool)
y_pred_orig = np.expm1(y_pred_log)
y_true_orig = np.expm1(test_r["Sales_log"])

rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
mae  = mean_absolute_error(y_true_orig, y_pred_orig)
print(f"RMSE: {rmse:,.0f}")
print(f"MAE:  {mae:,.0f}")

### Задание 11. Интерпретация CatBoost

In [ ]:
fi = pd.DataFrame({
    "feature": FEATURES,
    "importance": cb_model.get_feature_importance(),
}).nlargest(15, "importance").sort_values("importance")

plt.figure(figsize=(9, 6))
sns.barplot(x="importance", y="feature", data=fi, palette="viridis")
plt.title("Топ-15 важных признаков (CatBoost)")
plt.xlabel("Важность")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_true_orig, y_pred_orig, alpha=0.4, s=15, color="steelblue")
lim = [y_true_orig.min(), y_true_orig.max()]
plt.plot(lim, lim, "r--", lw=1.5, label="Идеальное предсказание")
plt.xlabel("Реальные продажи")
plt.ylabel("Предсказанные продажи")
plt.title("Predicted vs Actual Sales (CatBoost)")
plt.legend()
plt.tight_layout()
plt.show()

**Интерпретация:**

- **Самый важный признак** — `Store` (идентификатор магазина). Разные магазины имеют
  кардинально разные базовые уровни продаж: размер, расположение, лояльность клиентов.
  Именно между магазинами наибольшая дисперсия целевой переменной.

- **Scatter plot:** основное облако точек лежит близко к диагонали — модель хорошо
  калибрована. Отклонения выше диагонали означают недооценку (реальные продажи выше
  предсказанных), ниже — переоценку.

- **При каких продажах ошибки больше?** При высоких значениях продаж разброс точек
  вокруг диагонали шире — модель хуже предсказывает пиковые дни (акции, праздники).

### Задание 12. Анализ ошибок и итоговые выводы

In [ ]:
err_r = test_r[["Date", "Store", "StoreType", "Assortment", "Promo"]].copy()
err_r["y_true"] = y_true_orig.values
err_r["y_pred"] = y_pred_orig
err_r["abs_error"] = np.abs(err_r["y_true"] - err_r["y_pred"])

monthly_mae = (
    err_r.groupby(err_r["Date"].dt.to_period("M"))["abs_error"]
    .mean()
    .reset_index()
)
monthly_mae["Date"] = monthly_mae["Date"].astype(str)

plt.figure(figsize=(8, 4))
sns.barplot(x="Date", y="abs_error", data=monthly_mae, palette="Blues_d")
plt.title("Средний MAE по месяцам тестового периода")
plt.xlabel("Месяц")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

In [ ]:
store_err = (
    err_r.groupby("Store")
    .agg(mean_abs_error=("abs_error", "mean"),
         StoreType=("StoreType", "first"),
         Assortment=("Assortment", "first"),
         Promo=("Promo", "mean"))
    .nlargest(10, "mean_abs_error")
)
print("10 магазинов с наибольшим средним абсолютным отклонением:")
print(store_err.to_string())

**Итоговый вывод – сравнение трёх подходов:**

| Подход | Преимущество | Недостаток | Типичная область применения |
|---|---|---|---|
| Одиночное дерево | Интерпретируемо, быстро обучается, работает с категориями | Нестабильно, склонно к переобучению, низкое качество | Объяснимые решения, прототипирование, малые данные |
| Случайный лес | Стабилен, высокое качество «из коробки», встроенный OOB | Медленнее одиночного дерева, «чёрный ящик» | Табличные данные, задачи без временно́й зависимости |
| Градиентный бустинг (CatBoost) | Лучшее качество на табличных данных, нативные категории, временны́е ряды | Много гиперпараметров, риск переобучения без early stopping | Соревнования, production-системы, временны́е ряды |